## 🎯 과제 5 (선택): max_tokens로 답변 길이 제어 (실무 빈도 ★★★☆☆)

1교시에서 배운 `max_tokens` 파라미터를 적용해보세요. 토큰 폭주 방지와 비용 관리에 핵심입니다.

**요구사항**
- 모든 답변을 200 토큰 이내로 제한
- **힌트**: `ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=200)`

In [3]:
# 3️⃣ API 키 불러오기 (환경에 따라 자동 분기)
import os
import sys

# Colab에서 실행 중인지 자동 판별
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:

    from dotenv import load_dotenv
    load_dotenv()  # .env 파일에서 자동으로 불러옴

# 키가 잘 들어왔는지 확인 (앞 7글자만 출력 — 전체 노출은 위험)
api_key = os.environ.get("OPENAI_API_KEY", "")
print(f"API 키 로드: {api_key[:7]}..." if api_key else "❌ 키를 못 찾았어요")


API 키 로드: sk-proj...


In [2]:
# 1️⃣ OpenAI 클라이언트 생성 (한 번만 만들면 계속 재사용)
from openai import OpenAI

client = OpenAI()  # 환경변수의 OPENAI_API_KEY를 자동으로 읽어옴
print("✅ OpenAI 클라이언트 준비 완료")


✅ OpenAI 클라이언트 준비 완료


In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "{role}"),
    ("human", "{question}"),
])

In [5]:
# 페르소나 사전: 이름 → system 메시지
PERSONAS = {
    "분석가": {
        "system": "당신은 데이터 분석가입니다. 근거와 수치를 중심으로 논리적이고 간결하게 답변하세요.",
        "temperature": 0,
    },
    "디자이너": {
        "system": "당신은 UX/UI 디자이너입니다. 사용자 경험 관점에서 창의적이고 감각적으로 답변하세요.",
        "temperature": 0.8,
    },
}
print(f"✅ {len(PERSONAS)}개 페르소나 등록 완료: {list(PERSONAS.keys())}")

✅ 2개 페르소나 등록 완료: ['분석가', '디자이너']


In [6]:
def ask(persona_name: str, question: str) -> str:
    # 1. 페르소나 설정 조회 (system 문구 + temperature)
    config = PERSONAS[persona_name]

    # 2. 페르소나별 temperature를 적용한 체인 (prompt | llm 패턴)
    persona_chain = prompt | llm.bind(temperature=config["temperature"], max_tokens=200)

    # 3. 체인 호출 후 답변 텍스트만 반환
    result = persona_chain.invoke({"role": config["system"], "question": question})
    return result.content

In [7]:
# 명령어 처리 함수: (새로운_페르소나, 종료여부) 튜플 반환
def handle_command(user_input: str, current_persona: str):
    parts = user_input.split(maxsplit=1)   # "/change 마케터" → ["/change", "마케터"]
    cmd = parts[0]

    if cmd in ("/quit", "/exit"):
        print("👋 종료합니다.")
        return current_persona, True       # 종료 플래그 True

    if cmd == "/persona":
        print(f"🎭 현재 페르소나: {current_persona}")

    elif cmd == "/help":
        print("📖 명령어: /quit, /exit, /change <이름>, /persona, /help")

    elif cmd == "/change":
        new_name = parts[1] if len(parts) > 1 else ""
        if new_name in PERSONAS:
            print(f"✅ 페르소나가 '{new_name}'(으)로 변경되었습니다.")
            return new_name, False         # 페르소나만 변경
        else:
            print(f"❌ '{new_name}'은 등록된 페르소나가 아닙니다.")
            print(f"   사용 가능: {', '.join(PERSONAS.keys())}")

    else:
        print(f"❓ 알 수 없는 명령어: {cmd} (/help 입력)")

    return current_persona, False

In [8]:
# 챗봇 실행 함수
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("   /help 로 명령어 보기, /quit 로 종료\n")

    while True:
        user_input = input("질문> ").strip()
        if not user_input:                       # 빈 입력은 무시
            continue

        if user_input.startswith("/"):           # 명령어 처리
            persona, should_quit = handle_command(user_input, persona)
            if should_quit:
                break
        else:                                    # 일반 질문 처리
            answer = ask(persona, user_input)
            print(f"[{persona}] {answer}\n")

run_chatbot("분석가")

👋 페르소나 챗봇입니다. (현재: 분석가)
   /help 로 명령어 보기, /quit 로 종료

[분석가] 분석은 데이터를 수집, 정리, 해석하여 유의미한 정보를 도출하는 과정입니다. 이 과정은 다양한 방법론과 도구를 활용하여 데이터를 시각화하고, 패턴을 발견하며, 인사이트를 제공합니다. 분석의 정의는 크게 두 가지로 나눌 수 있습니다: 기술적 분석(데이터의 현재 상태를 이해)과 예측적 분석(미래의 경향을 예측).

분석의 중요성은 여러 측면에서 나타납니다. 첫째, 의사결정 지원입니다. 기업이나 조직은 분석을 통해 데이터 기반의 결정을 내릴 수 있으며, 이는 리스크를 줄이고 효율성을 높이는 데 기여합니다. 예를 들어, 2020년의 한 조사에 따르면, 데이터 기반 의사결정을 하는 기업은 그렇지 않은 기업보다 5~6% 더 높은 생산성을 기록했습니다.

둘째, 고객 이해입니다. 분석을

👋 종료합니다.
